# Ava Tokenizer Training

**SentencePiece BPE** ტოკენიზატორის ტრენინგი ქართულ კორპუსზე.

- **კორპუსი**: [CulturaX](https://huggingface.co/datasets/uonlp/CulturaX) — ქართული (`ka`) სექცია (~2.6B ტოკენი)
- **ალგორითმი**: BPE (Byte Pair Encoding) — სტანდარტი ავტორეგრესიული LM-ებისთვის
- **ლექსიკონი**: 32,000 ტოკენი
- **შედეგი**: HuggingFace-თავსებადი ტოკენიზატორი `data/tokenizer/` ფოლდერში

## 1. Dependencies

In [ ]:
# !pip install sentencepiece datasets transformers huggingface_hub

## 2. Configuration

In [ ]:
import os
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────
DATA_DIR = Path("../data")
CORPUS_FILE = DATA_DIR / "corpus_ka.txt"
TOKENIZER_DIR = DATA_DIR / "tokenizer"
TOKENIZER_PREFIX = TOKENIZER_DIR / "ava_ka_bpe"

# ── Tokenizer params ──────────────────────────────────────────
VOCAB_SIZE = 32_000
MAX_DOCS = 500_000          # None = use all (~3.1M docs, needs ~10GB+ RAM)
MAX_SENTENCE_LENGTH = 4096  # skip very long lines
INPUT_SENTENCE_SIZE = 500_000  # sentences to sample for training (RAM-friendly)

# ── Special tokens ────────────────────────────────────────────
PAD_TOKEN = "<|pad|>"
BOS_TOKEN = "<|bos|>"
EOS_TOKEN = "<|eos|>"
UNK_TOKEN = "<|unk|>"
USER_DEFINED_SYMBOLS = ["<|user|>", "<|ava|>", "<|enduser|>", "<|endava|>"]

os.makedirs(TOKENIZER_DIR, exist_ok=True)
print(f"Vocab size: {VOCAB_SIZE:,}")
print(f"Max docs: {MAX_DOCS or 'all'}")
print(f"Input sentence size: {INPUT_SENTENCE_SIZE:,}")
print(f"Output: {TOKENIZER_DIR}")

## 3. Download Corpus

CulturaX-ის ქართული (`ka`) ნაწილი — mC4 + OSCAR-ის გაწმენდილი სუპერსეტი.

> **შენიშვნა**: CulturaX მოითხოვს HuggingFace ავტორიზაციას. გაუშვით `huggingface-cli login` წინასწარ.

In [ ]:
from datasets import load_dataset

if CORPUS_FILE.exists():
    print(f"Corpus already exists: {CORPUS_FILE}")
    line_count = sum(1 for _ in open(CORPUS_FILE, encoding="utf-8"))
    print(f"Lines: {line_count:,}")
else:
    print("Downloading CulturaX Georgian subset...")
    ds = load_dataset(
        "uonlp/CulturaX",
        "ka",
        split="train",
        streaming=True,
        trust_remote_code=True,
    )

    count = 0
    with open(CORPUS_FILE, "w", encoding="utf-8") as f:
        for doc in ds:
            text = doc["text"].strip()
            if len(text) < 20:  # skip very short docs
                continue
            f.write(text + "\n")
            count += 1
            if count % 50_000 == 0:
                print(f"  {count:,} docs written...")
            if MAX_DOCS and count >= MAX_DOCS:
                break

    size_mb = CORPUS_FILE.stat().st_size / 1e6
    print(f"Done! {count:,} docs, {size_mb:.0f} MB -> {CORPUS_FILE}")

## 4. Train SentencePiece BPE Tokenizer

In [ ]:
import sentencepiece as spm

print(f"Training SentencePiece BPE tokenizer...")
print(f"  Corpus: {CORPUS_FILE}")
print(f"  Vocab:  {VOCAB_SIZE:,}")
print(f"  Sampling: {INPUT_SENTENCE_SIZE:,} sentences")
print()

spm.SentencePieceTrainer.train(
    input=str(CORPUS_FILE),
    model_prefix=str(TOKENIZER_PREFIX),
    vocab_size=VOCAB_SIZE,
    model_type="bpe",

    # Georgian has 33 letters — full coverage
    character_coverage=1.0,

    # Special tokens (IDs 0-3)
    pad_id=0,
    bos_id=1,
    eos_id=2,
    unk_id=3,
    pad_piece=PAD_TOKEN,
    bos_piece=BOS_TOKEN,
    eos_piece=EOS_TOKEN,
    unk_piece=UNK_TOKEN,

    # User-defined symbols (conversation markers)
    user_defined_symbols=USER_DEFINED_SYMBOLS,

    # RAM management
    input_sentence_size=INPUT_SENTENCE_SIZE,
    shuffle_input_sentence=True,
    train_extremely_large_corpus=True,

    # Quality settings
    byte_fallback=True,           # no UNK tokens ever
    split_digits=True,            # better number handling
    max_sentence_length=MAX_SENTENCE_LENGTH,
    num_threads=os.cpu_count(),
)

print(f"\nModel saved: {TOKENIZER_PREFIX}.model")
print(f"Vocab saved: {TOKENIZER_PREFIX}.vocab")

## 5. Verify Tokenizer

In [ ]:
sp = spm.SentencePieceProcessor()
sp.load(str(TOKENIZER_PREFIX) + ".model")

print(f"Vocab size: {sp.get_piece_size():,}")
print(f"PAD={sp.pad_id()}, BOS={sp.bos_id()}, EOS={sp.eos_id()}, UNK={sp.unk_id()}")
print()

test_texts = [
    "გამარჯობა, მე ვარ ავა — ქართული ენის მოდელი.",
    "ხელოვნური ინტელექტი არის კომპიუტერული მეცნიერების დარგი.",
    "Hello, I can also handle English text.",
    "Python 3.11 is 25% faster than 3.10.",
    "<|user|> რა არის მანქანური სწავლება? <|enduser|>",
]

for text in test_texts:
    tokens = sp.encode(text, out_type=str)
    ids = sp.encode(text)
    decoded = sp.decode(ids)
    print(f"Text:    {text}")
    print(f"Tokens:  {tokens}")
    print(f"IDs:     {ids}")
    print(f"Decoded: {decoded}")
    print(f"Count:   {len(ids)} tokens")
    print()

## 6. Fertility Analysis

ტოკენიზატორის ეფექტურობის შეფასება — რამდენ ტოკენს ხარჯავს სიტყვაზე (fertility).

კარგი ქართული ტოკენიზატორისთვის fertility უნდა იყოს **< 2.0**.

In [ ]:
import random

# Reservoir sampling — reads file line-by-line, never loads entire file in RAM
sample_size = 10_000
sample = []
with open(CORPUS_FILE, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i < sample_size:
            sample.append(line)
        else:
            j = random.randint(0, i)
            if j < sample_size:
                sample[j] = line

total_words = 0
total_tokens = 0
total_chars = 0

for line in sample:
    text = line.strip()
    if not text:
        continue
    words = text.split()
    tokens = sp.encode(text)
    total_words += len(words)
    total_tokens += len(tokens)
    total_chars += len(text)

fertility = total_tokens / total_words
compression = total_chars / total_tokens

print(f"Sample: {len(sample):,} documents (reservoir sampled)")
print(f"Words:  {total_words:,}")
print(f"Tokens: {total_tokens:,}")
print()
print(f"Fertility:   {fertility:.2f} tokens/word {'✓' if fertility < 2.0 else '✗ (too high)'}")
print(f"Compression: {compression:.2f} chars/token")

## 7. Convert to HuggingFace Format

SentencePiece მოდელის კონვერტაცია HuggingFace `PreTrainedTokenizerFast`-ში,
რომ `train.ipynb`-ში პირდაპირ გამოვიყენოთ.

In [ ]:
from ava.tokenizer import AvaTokenizer

tokenizer = AvaTokenizer(str(TOKENIZER_PREFIX) + ".model")

# Verify
print(f"Vocab size: {len(tokenizer):,}")
print(f"PAD={tokenizer.pad_token_id}, BOS={tokenizer.bos_token_id}, "
      f"EOS={tokenizer.eos_token_id}, UNK={tokenizer.unk_token_id}")
print()

test = "გამარჯობა, მე ვარ ავა!"
enc = tokenizer(test, return_tensors="pt")
print(f"Input:   {test}")
print(f"Tokens:  {tokenizer.tokenize(test)}")
print(f"IDs:     {enc['input_ids'].tolist()}")
print(f"Decoded: {tokenizer.decode(enc['input_ids'][0], skip_special_tokens=True)}")

In [ ]:
# Save tokenizer
tokenizer.save_pretrained(str(TOKENIZER_DIR))
print(f"Tokenizer saved to: {TOKENIZER_DIR}/")
print()
print("Files:")
for f in sorted(TOKENIZER_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f"  {f.name} ({size:.0f} KB)")

## 8. Usage in Training

`train.ipynb`-ში ტოკენიზატორის ჩატვირთვა:

```python
from ava.tokenizer import AvaTokenizer

tokenizer = AvaTokenizer.from_pretrained("data/tokenizer")

config.vocab_size = len(tokenizer)         # 32000
config.pad_token_id = tokenizer.pad_token_id
config.bos_token_id = tokenizer.bos_token_id
config.eos_token_id = tokenizer.eos_token_id
```